Chapter: Imbalance Learning <br>
Dataset: German Credit

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

---

Section 01: Inspecting Dataset

In [ ]:
df = pd.read_csv("credit.csv")
df

<small>Dataset Information</size>

In [ ]:
instances, features = df.shape
print("Number of Record = ", instances)
print("Number of features = ", features)
print("Column Name = ", df.columns.to_list())


<small>SEX</small>

In [ ]:
print(df['SEX'].value_counts())

<small>NO_OF_DEPENDENT</small>

In [ ]:
print(df['NO_OF_DEPENDENT'].describe())

type = df['NO_OF_DEPENDENT'].value_counts().index
height = df['NO_OF_DEPENDENT'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.grid()

<small>MARITAL_STATUS</small>

In [ ]:
print(df['MARITAL_STATUS'].value_counts())

type = df['MARITAL_STATUS'].value_counts().index
height = df['MARITAL_STATUS'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.rcParams['font.size'] = 6

<small>YRS_IN_PRESENT_JOB</small>

In [ ]:
print(df['YRS_IN_PRESENT_JOB'].describe())

type = df['YRS_IN_PRESENT_JOB'].value_counts().index
height = df['YRS_IN_PRESENT_JOB'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.grid()

<samll>AGE</small>

In [ ]:
print(df['AGE'].describe())

type = df['AGE'].value_counts().index
height = df['AGE'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.grid()

<small>EDUCATION</small>

In [ ]:
print(df['EDUCATION'].value_counts())

type = df['EDUCATION'].value_counts().index
height = df['EDUCATION'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.rcParams['font.size'] = 6

<small>YEARS_OF_EXPERIENCE</small>

In [ ]:
print(df['YEARS_OF_EXPERIENCE'].describe())

type = df['YEARS_OF_EXPERIENCE'].value_counts().index
height = df['YEARS_OF_EXPERIENCE'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.grid()

<small>TOTAL_MONTHLY_INCOME</small>

In [ ]:
print(df['TOTAL_MONTHLY_INCOME'].describe())

plt.hist(df['TOTAL_MONTHLY_INCOME'], bins=10)
plt.xlabel("Range")
plt.ylim(0, 100)
plt.ylabel("Frequency")
plt.grid();

<small>LOAN_AMOUNT</small>

In [ ]:
print(df['LOAN_AMOUNT'].describe())

plt.hist(df['LOAN_AMOUNT'], bins=50)
plt.xlabel("Range")
plt.ylabel("Frequency")
plt.grid();

<small>COMPANY_TYPE</small>

In [ ]:
print(df['COMPANY_TYPE'].value_counts())

type = df['COMPANY_TYPE'].value_counts().index
height = df['COMPANY_TYPE'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.rcParams['font.size'] = 6

<small>LABEL</small>

In [ ]:
print(df['LABEL'].value_counts())
print(df['LABEL'].value_counts(normalize=True) * 100)


type = df['LABEL'].value_counts().index
height = df['LABEL'].value_counts().values
plt.bar(type, height)
plt.xlabel("Type")
plt.ylabel("Frequency")
plt.rcParams['font.size'] = 6

---

Section 02: Preprocessing data

In [ ]:
df = pd.read_csv("credit.csv")
df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Load dataset
df = pd.read_csv("credit.csv")

#---------- Remove Identifier ----------#
df = df.drop(columns=["No"])


#---------- Integer and Float ----------#
cols = [
    "NO_OF_DEPENDENT",
    "YRS_IN_PRESENT_JOB",
    "AGE",
    "YEARS_OF_EXPERIENCE",
    "TOTAL_MONTHLY_INCOME",
    "LOAN_AMOUNT"
]

df[cols] = MinMaxScaler().fit_transform(df[cols])


#---------- Binary Encoding ----------#
df["SEX"] = df["SEX"].map({
    "FEMALE": 0,
    "MALE": 1
})


#---------- One-Hot Encoding ----------#
df = pd.get_dummies(
    df,
    columns=[
        "MARITAL_STATUS",
        "EDUCATION",
        "COMPANY_TYPE"
    ],
    dtype=int
)


#---------- Boolean Target ----------#
df["LABEL"] = df["LABEL"].astype(int)


#---------- Check Result ----------#
df.head()


In [ ]:
df.to_csv("credit_pre.csv", index=False)

---

Section 03: Separate training data and Test data

In [ ]:
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('credit_pre.csv')

# Split X, y
X = df.drop(columns='LABEL')
y = df['LABEL']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

pd.DataFrame({
    'Original (%)': y.value_counts(normalize=True) * 100,
    'Train (%)': y_train.value_counts(normalize=True) * 100,
    'Test (%)': y_test.value_counts(normalize=True) * 100
}).sort_index()


In [ ]:
# Save files
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

---

Section 04: No Sampling (Benchmark)

<small>Decision Tree</small>

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

#---------- Load data ----------#
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()



#---------- Parameter tuning ----------#
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(criterion='gini', random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Best model
best_dt = grid.best_estimator_

print("Best Parameters")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.4f}")



#---------- Predict ----------#
y_pred = best_dt.predict(X_test)

# Metrics
print("\n===== Test Result =====")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}")


cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No', 'Yes']
)
disp.plot(
    cmap='Blues',
    ax=ax,
    values_format='d',
    colorbar=True
)
# Increase font sizes
ax.set_title('Decision Tree Confusion Matrix', fontsize=18, pad=15)
ax.set_xlabel('Predicted label', fontsize=15, labelpad=10)
ax.set_ylabel('True label', fontsize=15, labelpad=10)
ax.tick_params(axis='both', labelsize=14)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

# Increase colorbar font size
cbar = disp.im_.colorbar
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


<small>Random Forest</small>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

#---------- Load data ----------#
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()


#---------- Parameter tuning ----------#
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)


#---------- Best model ----------#
best_rf = grid.best_estimator_

print("Best Parameters")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.4f}")


#---------- Predict ----------#
y_pred = best_rf.predict(X_test)


#---------- Metrics ----------#
print("\n===== Test Result =====")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")


#---------- Confusion Matrix ----------#
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No', 'Yes']
)

disp.plot(
    cmap='Blues',
    ax=ax,
    values_format='d',
    colorbar=True
)

# Increase font sizes
ax.set_title('Random Forest Confusion Matrix',
             fontsize=18, pad=15)
ax.set_xlabel('Predicted label',
              fontsize=15, labelpad=10)
ax.set_ylabel('True label',
              fontsize=15, labelpad=10)
ax.tick_params(axis='both', labelsize=14)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

# Increase colorbar font size
cbar = disp.im_.colorbar
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


Section 05: Using Under Sampling

<small>Decision Tree</small>

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline


#---------- Load data ----------#
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()


#---------- Pipeline ----------#
pipeline = Pipeline([
    ('undersampler', RandomUnderSampler(random_state=42)),
    ('dt', DecisionTreeClassifier(
        criterion='gini',
        random_state=42
    ))
])


#---------- Parameter tuning ----------#
param_grid = {
    'dt__max_depth': [3, 5, 7, 10, None],
    'dt__min_samples_split': [2, 5, 10],
    'dt__min_samples_leaf': [1, 2, 5]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)


#---------- Best model ----------#
best_dt = grid.best_estimator_

print("Best Parameters")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.4f}")


#---------- Predict ----------#
y_pred = best_dt.predict(X_test)


#---------- Metrics ----------#
print("\n===== Test Result =====")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")


#---------- Confusion Matrix ----------#
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No', 'Yes']
)

disp.plot(
    cmap='Blues',
    ax=ax,
    values_format='d',
    colorbar=True
)

# Increase font sizes
ax.set_title(
    'Decision Tree with Random Undersampling',
    fontsize=18, pad=15
)
ax.set_xlabel('Predicted label', fontsize=15, labelpad=10)
ax.set_ylabel('True label', fontsize=15, labelpad=10)
ax.tick_params(axis='both', labelsize=14)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

# Increase colorbar font size
cbar = disp.im_.colorbar
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


<small>Random Forest</small>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline


#---------- Load data ----------#
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()


#---------- Pipeline ----------#
pipeline = Pipeline([
    ('undersampler', RandomUnderSampler(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=1))
])


#---------- Parameter tuning ----------#
param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [5, 10, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)


#---------- Best model ----------#
best_rf = grid.best_estimator_

print("Best Parameters")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.4f}")


#---------- Predict ----------#
y_pred = best_rf.predict(X_test)


#---------- Metrics ----------#
print("\n===== Test Result =====")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")


#---------- Confusion Matrix ----------#
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No', 'Yes']
)

disp.plot(
    cmap='Blues',
    ax=ax,
    values_format='d',
    colorbar=True
)

# Increase font sizes
ax.set_title(
    'Random Forest with Random Undersampling',
    fontsize=18, pad=15
)
ax.set_xlabel('Predicted label', fontsize=15, labelpad=10)
ax.set_ylabel('True label', fontsize=15, labelpad=10)
ax.tick_params(axis='both', labelsize=14)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

# Increase colorbar font size
cbar = disp.im_.colorbar
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


---

Section 06: Using Over Sampling

<small>Decision Tree</small>

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline


#---------- Load data ----------#
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()


#---------- Pipeline ----------#
pipeline = Pipeline([
    ('oversampler', RandomOverSampler(random_state=42)),
    ('dt', DecisionTreeClassifier(
        criterion='gini',
        random_state=42
    ))
])


#---------- Parameter tuning ----------#
param_grid = {
    'dt__max_depth': [3, 5, 7, 10, None],
    'dt__min_samples_split': [2, 5, 10],
    'dt__min_samples_leaf': [1, 2, 5]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)


#---------- Best model ----------#
best_dt = grid.best_estimator_

print("Best Parameters")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.4f}")


#---------- Predict ----------#
y_pred = best_dt.predict(X_test)


#---------- Metrics ----------#
print("\n===== Test Result =====")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")


#---------- Confusion Matrix ----------#
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No', 'Yes']
)

disp.plot(
    cmap='Blues',
    ax=ax,
    values_format='d',
    colorbar=True
)

# Increase font sizes
ax.set_title(
    'Decision Tree with Random Oversampling',
    fontsize=18, pad=15
)
ax.set_xlabel('Predicted label', fontsize=15, labelpad=10)
ax.set_ylabel('True label', fontsize=15, labelpad=10)
ax.tick_params(axis='both', labelsize=14)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

# Increase colorbar font size
cbar = disp.im_.colorbar
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


<small>Random Forest</small>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline


#---------- Load data ----------#
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()


#---------- Pipeline ----------#
pipeline = Pipeline([
    ('oversampler', RandomOverSampler(random_state=42)),
    ('rf', RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])


#---------- Parameter tuning ----------#
param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [5, 10, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)


#---------- Best model ----------#
best_rf = grid.best_estimator_

print("Best Parameters")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.4f}")


#---------- Predict ----------#
y_pred = best_rf.predict(X_test)


#---------- Metrics ----------#
print("\n===== Test Result =====")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")


#---------- Confusion Matrix ----------#
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No', 'Yes']
)

disp.plot(
    cmap='Blues',
    ax=ax,
    values_format='d',
    colorbar=True
)

# Increase font sizes
ax.set_title(
    'Random Forest with Random Oversampling',
    fontsize=18, pad=15
)
ax.set_xlabel('Predicted label', fontsize=15, labelpad=10)
ax.set_ylabel('True label', fontsize=15, labelpad=10)
ax.tick_params(axis='both', labelsize=14)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

# Increase colorbar font size
cbar = disp.im_.colorbar
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


---